# Fine-tune ViT5-base cho tóm tắt tin tức tiếng Việt

Notebook này chạy độc lập trên local, Kaggle Notebook hoặc Google Colab. Mục tiêu là fine-tune `VietAI/vit5-base` trên dataset `ithieund/VietNews-Abs-Sum` cho task tóm tắt văn bản tin tức tiếng Việt.

Gợi ý sử dụng:
- Chạy thử trước với `MAX_TRAIN_SAMPLES = 1000` và `MAX_EVAL_SAMPLES = 200`.
- Khi pipeline ổn, tăng số mẫu hoặc bật `FULL_DATASET = True`.
- Trên Kaggle/Colab nên bật GPU trước khi chạy notebook.

## 1. Cài thư viện

Nếu chạy local và đã cài `requirements.txt`, có thể bỏ qua cell này. Trên Kaggle/Colab, cell này sẽ cài các thư viện cần thiết cho training.

In [ ]:
import os
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = os.path.exists("/kaggle/working")

if IN_COLAB or IN_KAGGLE:
    %pip install -q "transformers>=4.40,<5.0" datasets evaluate rouge_score sentencepiece accelerate protobuf safetensors tqdm
else:
    print("Local runtime detected. Nếu thiếu thư viện, hãy chạy: pip install -r requirements.txt")

## 2. Kiểm tra runtime và cấu hình đường dẫn

In [ ]:
from pathlib import Path
import os
import random
import sys

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if "IN_COLAB" not in globals():
    try:
        import google.colab  # type: ignore
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
if "IN_KAGGLE" not in globals():
    IN_KAGGLE = os.path.exists("/kaggle/working")

if IN_KAGGLE:
    WORK_DIR = Path("/kaggle/working")
elif IN_COLAB:
    WORK_DIR = Path("/content")
else:
    WORK_DIR = Path.cwd().resolve()

MODEL_OUTPUT_DIR = WORK_DIR / "vit5-vietnews-summarization"
EVAL_OUTPUT_DIR = WORK_DIR / "vit5-vietnews-evaluation"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Work dir:", WORK_DIR)
print("Model output:", MODEL_OUTPUT_DIR)

## 3. Cấu hình thí nghiệm

Các giá trị mặc định bên dưới ưu tiên chạy được trên GPU phổ thông. Nếu bị out-of-memory, giảm `BATCH_SIZE` xuống 1 hoặc bật `GRADIENT_CHECKPOINTING = True`.

In [ ]:
MODEL_NAME = "VietAI/vit5-base"
DATASET_NAME = "ithieund/VietNews-Abs-Sum"
SOURCE_COLUMN = "article"
TARGET_COLUMN = "abstract"

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128
NUM_BEAMS = 4

EPOCHS = 1
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

MAX_TRAIN_SAMPLES = 1000
MAX_EVAL_SAMPLES = 200
MAX_TEST_SAMPLES = 200
FULL_DATASET = False

FP16 = torch.cuda.is_available()
GRADIENT_CHECKPOINTING = False

print({
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "fp16": FP16,
    "full_dataset": FULL_DATASET,
})

## 4. Tải và xem dataset

In [ ]:
from datasets import DatasetDict, load_dataset

raw_dataset = load_dataset(DATASET_NAME)
raw_dataset

In [ ]:
sample = raw_dataset["train"][0]
print("TITLE:\n", sample.get("title", ""))
print("\nARTICLE:\n", sample[SOURCE_COLUMN][:1200])
print("\nABSTRACT:\n", sample[TARGET_COLUMN])

## 5. Tokenize dữ liệu

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
print("Tokenizer vocab size:", len(tokenizer))

In [ ]:
def maybe_subset(split, max_samples, seed=SEED):
    if max_samples is None:
        return split
    max_samples = min(max_samples, len(split))
    return split.shuffle(seed=seed).select(range(max_samples))


def preprocess_batch(examples):
    model_inputs = tokenizer(
        examples[SOURCE_COLUMN],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples[TARGET_COLUMN],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_split = raw_dataset["train"] if FULL_DATASET else maybe_subset(raw_dataset["train"], MAX_TRAIN_SAMPLES)
eval_split = maybe_subset(raw_dataset["validation"], MAX_EVAL_SAMPLES)
test_split = maybe_subset(raw_dataset["test"], MAX_TEST_SAMPLES)

selected_dataset = DatasetDict({
    "train": train_split,
    "validation": eval_split,
    "test": test_split,
})

tokenized_dataset = selected_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)

tokenized_dataset

## 6. Khai báo metric ROUGE

In [ ]:
import evaluate

rouge = evaluate.load("rouge")


def compute_rouge_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    pad_token_id = tokenizer.pad_token_id

    predictions = np.where(predictions >= 0, predictions, pad_token_id)
    labels = np.where(labels != -100, labels, pad_token_id)

    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=False,
    )
    return {key: round(value * 100, 4) for key, value in scores.items()}

## 7. Fine-tune model

In [ ]:
import inspect
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_kwargs = {
    "output_dir": str(MODEL_OUTPUT_DIR),
    "overwrite_output_dir": True,
    "num_train_epochs": EPOCHS,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "fp16": FP16,
    "save_strategy": "epoch",
    "logging_steps": 50,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "predict_with_generate": True,
    "generation_max_length": MAX_TARGET_LENGTH,
    "generation_num_beams": NUM_BEAMS,
    "dataloader_num_workers": 0,
    "remove_unused_columns": False,
    "report_to": "none",
    "seed": SEED,
}

signature = inspect.signature(Seq2SeqTrainingArguments.__init__)
if "eval_strategy" in signature.parameters:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**training_kwargs)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_rouge_metrics,
)

train_result = trainer.train()
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))
train_result

## 8. Đánh giá trên tập test

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=tokenized_dataset["test"], metric_key_prefix="test")
for key, value in test_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

## 9. Thử sinh tóm tắt

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_OUTPUT_DIR))
inference_model = AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_OUTPUT_DIR)).to(device)
inference_model.eval()


def summarize(text, min_length=30, max_length=MAX_TARGET_LENGTH, num_beams=NUM_BEAMS):
    inputs = inference_tokenizer(
        text,
        return_tensors="pt",
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        summary_ids = inference_model.generate(
            **inputs,
            max_length=max_length,
            min_length=min_length,
            num_beams=num_beams,
            early_stopping=True,
        )
    return inference_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


example = raw_dataset["test"][0]
print("ARTICLE:\n", example[SOURCE_COLUMN][:1500])
print("\nREFERENCE:\n", example[TARGET_COLUMN])
print("\nPREDICTION:\n", summarize(example[SOURCE_COLUMN]))

## 10. Nén model để tải về hoặc lưu output

Trên Kaggle, file zip sẽ nằm trong `/kaggle/working`. Trên Colab, có thể tải file zip về máy từ panel Files hoặc dùng cell tải xuống bên dưới.

In [ ]:
import shutil

zip_base = WORK_DIR / "vit5-vietnews-summarization"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(MODEL_OUTPUT_DIR))
print("Saved:", zip_path)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print("Không chạy trên Colab. File zip đã được lưu tại:", zip_path)

## 11. Gợi ý chạy full training

Sau khi chạy thử thành công, quay lại cell cấu hình và đổi:

```python
EPOCHS = 3
MAX_TRAIN_SAMPLES = 20000
MAX_EVAL_SAMPLES = 2000
MAX_TEST_SAMPLES = 2000
```

Nếu muốn train toàn bộ tập train:

```python
FULL_DATASET = True
EPOCHS = 3
```

Với GPU ít VRAM, dùng:

```python
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
GRADIENT_CHECKPOINTING = True
```